In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

In [2]:
import pandas as pd

# Wider pandas display so the wide voting matrix prints fully in the notebook
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

# SF1 Audience Results Verification

This notebook reproduces the **Eurovision SF1 audience-voting calculation** end-to-end so the official scoring can be independently verified.

The pipeline is structured in stages:

1. **Stage 1 - Data structure**: load the input CSVs, define static reference data (pots, running order, ISO codes), and run basic data-quality checks.
2. **Stage 2 - Selection & cleanup**: pick which show (SF1 / SF2 / GF) to analyse, aggregate audience votes per country, and zero-out any self-voting.
3. **Stage 3 - Pot calculation**: detect countries below the audience-vote threshold and replace their votes by an equalised pot mean (or by jury votes, as fallback).
4. **Stage 4 - Jury calculation**: convert jury ranks into Eurovision points, apply tie-breaking, compute totals and global rankings.
5. **Stage 5 - Televoting calculation**: convert audience votes to points, apply tie-breaking, and produce the final televoting ranking.
6. **Final results**: combine jury + televoting into the official combined scoreboard and export Excel deliverables.

---

## Stage 1: Create the data structure

What this stage does:

- Import libraries and configure pandas display options.
- Define static reference dictionaries (CSV paths, pots, running order, ISO codes).
- Load all available Jury and Televoting CSVs into dataframes (missing files are skipped).
- Define the audience-vote threshold used in pot calculations.
- Run the first data-quality check: detect duplicated `TPartnerId` values per televoting provider.

In [3]:
import os
import shutil
import stat
import hashlib
import pandas as pd
from sqlalchemy import create_engine
import math
import numpy as np

########################
### STATIC VARIABLES ###
########################

SOURCE_CSV_DIRECTORY = "./Input/Test Scenario"
SCRIPT_XLSX_DIRECTORY = "./Output"

###########
# SF1 CSV #
###########

SF1_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-01_SF1_EY_Jury.csv")
SF1_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-01_SF1_EY_Televoting.csv")

###########
# SF2 CSV #
###########

SF2_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")
SF2_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")

##########
# GF CSV #
##########

GF_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")
GF_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")

################
# POTS MAPPING #
################

DICT_POTS_SF1 = {
    "Pot01": ["Croatia", "Finland", "Montenegro", "Serbia", "Sweden"],
    "Pot02": ["Belgium", "Georgia", "Israel", "Moldova", "Poland"],
    "Pot03": ["Estonia", "Greece", "Lithuania", "Portugal", "San Marino"],
}
 
DICT_POTS_SF2 = {
    "Pot01": ["Albania", "Australia", "Denmark", "Norway", "Switzerland"],
    "Pot02": ["Armenia", "Azerbaijan", "Luxembourg", "Romania", "Ukraine"],
    "Pot03": ["Bulgaria", "Cyprus", "Czechia", "Latvia", "Malta"],
}
 
DICT_POTS_GF = {
    "Pot01": ["Albania", "Bulgaria", "Croatia", "Montenegro", "Serbia", "Switzerland"],
    "Pot02": ["Australia", "Denmark", "Estonia", "Finland", "Norway", "Sweden"],
    "Pot03": ["Armenia", "Azerbaijan", "Georgia", "Israel", "Poland", "Ukraine"],
    "Pot04": ["Belgium", "Czechia", "Luxembourg", "Moldova", "Portugal", "Romania"],
    "Pot05": ["Cyprus", "Greece", "Latvia", "Lithuania", "Malta", "San Marino"]
}

DICT_POTS_PREQUALIFIED = {
    "Pot00": ["Austria", "France", "Germany", "Italy", "United Kingdom"]
}
 
LIST_POT_00_PREQUALIFIED = ["Austria", "France", "Germany", "Italy", "Rest Of World", "United Kingdom"]

#########################
# RUNNING ORDER MAPPING #
#########################

DICT_DDI_SF1 = {
    "DDI01": "Moldova",
    "DDI02": "Sweden",
    "DDI03": "Croatia",
    "DDI04": "Greece",
    "DDI05": "Portugal",
    "DDI06": "Georgia",
    "DDI07": "Finland",
    "DDI08": "Montenegro",
    "DDI09": "Estonia",
    "DDI10": "Israel",
    "DDI11": "Belgium",
    "DDI12": "Lithuania",
    "DDI13": "San Marino",
    "DDI14": "Poland",
    "DDI15": "Serbia"
}

DICT_DDI_SF2 = {
    "DDI01": "Bulgaria",
    "DDI02": "Azerbaijan",
    "DDI03": "Romania",
    "DDI04": "Luxembourg",
    "DDI05": "Czechia",
    "DDI06": "Armenia",
    "DDI07": "Switzerland",
    "DDI08": "Cyprus",
    "DDI09": "Latvia",
    "DDI10": "Denmark",
    "DDI11": "Australia",
    "DDI12": "Ukraine",
    "DDI13": "Albania",
    "DDI14": "Malta",
    "DDI15": "Norway"
}

DICT_DDI_GF = {
    # TO BE DEFINED THURSDAY EOB
}

DICT_ISO = {
    "Albania": "AL",
    "Andorra": "AD",
    "Armenia": "AM",
    "Australia": "AU",
    "Austria": "AT",
    "Azerbaijan": "AZ",
    "Belarus": "BY",
    "Belgium": "BE",
    "Bosnia & Herzegovina": "BA",
    "Bulgaria": "BG",
    "Croatia": "HR",
    "Cyprus": "CY",
    "Czechia": "CZ",
    "Denmark": "DK",
    "Estonia": "EE",
    "Finland": "FI",
    "France": "FR",
    "Georgia": "GE",
    "Germany": "DE",
    "Greece": "GR",
    "Hungary": "HU",
    "Iceland": "IS",
    "Ireland": "IE",
    "Israel": "IL",
    "Italy": "IT",
    "Latvia": "LV",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Malta": "MT",
    "Moldova": "MD",
    "Monaco": "MC",
    "Montenegro": "ME",
    "Morocco": "MA",
    "Netherlands": "NL",
    "North Macedonia": "MK",
    "Norway": "NO",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Russia": "RU",
    "San Marino": "SM",
    "Serbia": "RS",
    "Slovakia": "SK",
    "Slovenia": "SI",
    "Spain": "ES",
    "Sweden": "SE",
    "Switzerland": "CH",
    "Turkey": "TR",
    "Ukraine": "UA",
    "United Kingdom": "GB",
    "Rest Of World": "RoW",
}

MAPPING_DF_TO_DICT_DDI = {
    "sf1_jury_df": DICT_DDI_SF1,
    "sf1_televoting_df": DICT_DDI_SF1,
    "sf2_jury_df": DICT_DDI_SF2,
    "sf2_televoting_df": DICT_DDI_SF2,
    "gf_jury_df": DICT_DDI_GF,
    "gf_televoting_df": DICT_DDI_GF,
}

MAPPING_DF_TO_DICT_POTS = {
    "sf1_jury_df": DICT_POTS_SF1,
    "sf1_televoting_df": DICT_POTS_SF1,
    "sf2_jury_df": DICT_POTS_SF2,
    "sf2_televoting_df": DICT_POTS_SF2,
    "gf_jury_df": DICT_POTS_GF,
    "gf_televoting_df": DICT_POTS_GF,
}

POT_VOTE_COUNT_THRESHOLD = 1000

# list of countries to replace audience voting by jury voting
list_replace_audience_by_jury = []

#########################
### GENERAL FUNCTIONS ###
#########################

# Lists all user-created dataframes
def get_all_dataframe_names():
    dataframe_names = []
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            dataframe_names.append(name)
    return dataframe_names

####################
### QC FUNCTIONS ###
####################

def qc01_duplicated_tpartnerid_values():
    # Get the list of all user-created DataFrames
    user_created_dataframe_names = get_all_dataframe_names()

    # list of dataframes that are expected to contain the TPartnerId field (televoting only)
    televoting_dataframes = [
        "sf1_televoting_df",
        "sf2_televoting_df",
        "gf_televoting_df",
    ]

    for df_name in user_created_dataframe_names:
        if df_name in televoting_dataframes:
            df = eval(
                df_name
            )  # load dataframe into temporary variable based on original variable name
            duplicated_rows = df[
                df.duplicated(subset=["strName", "TPartnerId"], keep=False)
            ]

            if len(duplicated_rows) > 0:
                print(
                    f"\n### ERROR, {len(duplicated_rows)} duplicated [TPartnerId] values identified in dataframe {df_name}"
                )
                print(duplicated_rows[["strName", "TPartnerId"]])
            elif len(duplicated_rows) == 0:
                print(
                    f"\nNo duplicated [TPartnerId] values identified in dataframe {df_name}"
                )
            else:
                raise ValueError(
                    f"Unexpected value during qc01_duplicated_tpartnerid_values for dataframe {df_name}."
                )

#########################
### MAIN SCRIPT START ###
#########################

# Store all provided CSVs to dataframes
try:
    sf1_jury_df = pd.read_csv(SF1_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF1 Jury CSV file, skipping...")
try:
    sf1_televoting_df = pd.read_csv(SF1_TELEVOTING_CSV_FILE_PATH, sep=";")
except:
    print("\n### Missing SF1 Televoting CSV file, skipping...")
try:
    sf2_jury_df = pd.read_csv(SF2_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF2 Jury CSV file, skipping...")
try:
    sf2_televoting_df = pd.read_csv(SF2_TELEVOTING_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF2 Televoting CSV file, skipping...")
try:
    gf_jury_df = pd.read_csv(GF_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing GF Jury CSV file, skipping...")
try:
    gf_televoting_df = pd.read_csv(GF_TELEVOTING_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing GF Televoting CSV file, skipping...")
    
qc01_duplicated_tpartnerid_values()


### Missing SF2 Jury CSV file, skipping...

### Missing SF2 Televoting CSV file, skipping...

### Missing GF Jury CSV file, skipping...

### Missing GF Televoting CSV file, skipping...

No duplicated [TPartnerId] values identified in dataframe sf1_televoting_df


---

---

## Stage 2a: Selecting the dataset to analyse

Pick which show the Rest Of the notebook should run for (SF1 / SF2 / Grand Final televoting dataframe). The selected name drives:

- which DDI-to-country mapping is used,
- which pot mapping is used,
- which Jury dataframe is paired with the televoting one downstream.

In [4]:
###############################
### SELECTING DF TO ANALYZE ###
###############################

########################################
selected_df_name = "sf1_televoting_df"
# selected_df_name = 'sf2_televoting_df'
# selected_df_name = 'gf_televoting_df'
########################################

selected_df = eval(selected_df_name)

# Identify corresponding dictionary
corresponding_ddi_dict = MAPPING_DF_TO_DICT_DDI[selected_df_name]
corresponding_pots_dict = MAPPING_DF_TO_DICT_POTS[selected_df_name]

print(
    f"\nAnalysis running for dataframe =>***{selected_df_name.upper()}***<=\n\nDDI mapping: {corresponding_ddi_dict}\n\n"
)


Analysis running for dataframe =>***SF1_TELEVOTING_DF***<=

DDI mapping: {'DDI01': 'Moldova', 'DDI02': 'Sweden', 'DDI03': 'Croatia', 'DDI04': 'Greece', 'DDI05': 'Portugal', 'DDI06': 'Georgia', 'DDI07': 'Finland', 'DDI08': 'Montenegro', 'DDI09': 'Estonia', 'DDI10': 'Israel', 'DDI11': 'Belgium', 'DDI12': 'Lithuania', 'DDI13': 'San Marino', 'DDI14': 'Poland', 'DDI15': 'Serbia'}




---

---

### Aggregating audience votes per country

Group the raw televoting rows by `strName` (country code) and sum every `nVotesforDDI*` column, so that we end up with one aggregated row per voting country.

In [5]:
# Keep only the per-DDI vote columns
filtered_columns = [col for col in selected_df.columns if col.startswith("nVotesforDDI")]

# Group rows by country code (strName) and sum the per-DDI vote counts
grouped_df = selected_df.groupby("strName")[filtered_columns].sum()

# Re-add the partner id placeholder for downstream consistency
grouped_df.insert(0, "TPartnerId", "AGGREGATED")
grouped_df = grouped_df.reset_index(drop=False)

grouped_df

,strName,TPartnerId,nVotesforDDI01,nVotesforDDI02,nVotesforDDI03,nVotesforDDI04,nVotesforDDI05,nVotesforDDI06,nVotesforDDI07,nVotesforDDI08,nVotesforDDI09,nVotesforDDI10,nVotesforDDI11,nVotesforDDI12,nVotesforDDI13,nVotesforDDI14,nVotesforDDI15
0,BE,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,91422,0,74529,60685,97257,12001
1,DE,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790
2,EE,AGGREGATED,61921,103959,109643,95444,42056,55468,34756,38263,0,90089,37495,114399,23581,81317,26804
3,FI,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239
4,GE,AGGREGATED,22623,75601,46846,9905,3750,0,23166,24521,65585,49739,28214,28763,75289,59033,6976
5,GR,AGGREGATED,84356,16916,16659,0,93256,36604,79944,26493,43248,81486,98868,33928,73547,57259,20203
6,HR,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019
7,IL,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,0,37495,114399,23581,81317,26804
8,IT,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456
9,LT,AGGREGATED,8954,3643,18035,8169,16090,16502,8592,14772,15620,10714,15556,0,3771,571,16555


---

---

### Adding readable country names

- Map the ISO code in `strName` to the full country name (`FullCountryName`).
- Append the recipient country name to each `nVotesforDDI##` column header so the matrix becomes self-describing.

In [6]:
# 1) Add the full country name resolved from the ISO code (strName)
inverted_DICT_ISO = {v: k for k, v in DICT_ISO.items()}
grouped_df_w_full_country_names = grouped_df.assign(
    FullCountryName=grouped_df["strName"].map(inverted_DICT_ISO)
)
grouped_df_w_full_country_names.insert(
    0, "FullCountryName", grouped_df_w_full_country_names.pop("FullCountryName")
)

# 2) Append the recipient country name to each nVotesforDDI## column header,
#    so a column reads e.g. nVotesforDDI01_Moldova instead of nVotesforDDI01.
for ddi_key, country_name in corresponding_ddi_dict.items():
    ddi_number = ddi_key[-2:]
    column_to_update = [
        col for col in grouped_df_w_full_country_names.columns
        if col.startswith(f"nVotesforDDI{ddi_number}")
    ][0]
    new_column_name = f"{column_to_update}_{country_name}"
    grouped_df_w_full_country_names = grouped_df_w_full_country_names.rename(
        columns={column_to_update: new_column_name}
    )

grouped_df_w_full_country_names

,FullCountryName,strName,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia
0,Belgium,BE,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,91422,0,74529,60685,97257,12001
1,Germany,DE,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790
2,Estonia,EE,AGGREGATED,61921,103959,109643,95444,42056,55468,34756,38263,0,90089,37495,114399,23581,81317,26804
3,Finland,FI,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239
4,Georgia,GE,AGGREGATED,22623,75601,46846,9905,3750,0,23166,24521,65585,49739,28214,28763,75289,59033,6976
5,Greece,GR,AGGREGATED,84356,16916,16659,0,93256,36604,79944,26493,43248,81486,98868,33928,73547,57259,20203
6,Croatia,HR,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019
7,Israel,IL,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,0,37495,114399,23581,81317,26804
8,Italy,IT,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456
9,Lithuania,LT,AGGREGATED,8954,3643,18035,8169,16090,16502,8592,14772,15620,10714,15556,0,3771,571,16555


---

---

### Sanity check: full country names populated

If any `FullCountryName` remained `NaN` after the ISO mapping (e.g. for `Rest Of World`), fall back to the original `strName` value so no row is left without a label.

In [7]:
# Fallback: if any FullCountryName is missing, use the original ISO code as label
if any(grouped_df_w_full_country_names["FullCountryName"].isna()):
    grouped_df_w_full_country_names["FullCountryName"] = grouped_df_w_full_country_names["strName"]

---

---

### Detecting and neutralising self-voting

A country is not allowed to vote for itself. We scan the matrix and force any non-zero self-voting cell to `0`, printing an error so the issue is visible in the run log.

In [8]:
# A country must not vote for itself: any non-zero self-vote cell is reset to 0
for index, row in grouped_df_w_full_country_names.iterrows():
    full_country_name = row["FullCountryName"]
    matching_column = [
        col for col in grouped_df_w_full_country_names.columns
        if col.startswith("nVotesforDDI") and col.endswith(full_country_name)
    ]

    if matching_column:
        current_value = grouped_df_w_full_country_names.at[index, matching_column[0]]
        if current_value != 0:
            print(f"### ERROR: SELF-VOTING COUNTRY IDENTIFIED! Country: {full_country_name} - Value: {current_value}")
            grouped_df_w_full_country_names.at[index, matching_column[0]] = 0

grouped_df_w_full_country_names

,FullCountryName,strName,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia
0,Belgium,BE,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,91422,0,74529,60685,97257,12001
1,Germany,DE,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790
2,Estonia,EE,AGGREGATED,61921,103959,109643,95444,42056,55468,34756,38263,0,90089,37495,114399,23581,81317,26804
3,Finland,FI,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239
4,Georgia,GE,AGGREGATED,22623,75601,46846,9905,3750,0,23166,24521,65585,49739,28214,28763,75289,59033,6976
5,Greece,GR,AGGREGATED,84356,16916,16659,0,93256,36604,79944,26493,43248,81486,98868,33928,73547,57259,20203
6,Croatia,HR,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019
7,Israel,IL,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,0,37495,114399,23581,81317,26804
8,Italy,IT,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456
9,Lithuania,LT,AGGREGATED,8954,3643,18035,8169,16090,16502,8592,14772,15620,10714,15556,0,3771,571,16555


---

---

### Assigning a Pot to each voting country

Every country is mapped to its allocated pot (`Pot01`-`Pot03` for SF1, plus `Pot00` for the prequalified Big Five / host).

### Reference: list of pre-qualified countries (Pot00)

Quick visual check of the prequalified country list.


In [9]:
LIST_POT_00_PREQUALIFIED

['Austria', 'France', 'Germany', 'Italy', 'Rest Of World', 'United Kingdom']

In [10]:
# Build a country -> pot mapping from the active pot dictionary
inverted_DICT_POTS = {
    country: pot
    for pot, countries in corresponding_pots_dict.items()
    for country in countries
}
pot_df = grouped_df_w_full_country_names

# Add (and reposition) the POT column
pot_df["POT"] = pot_df["FullCountryName"].map(inverted_DICT_POTS)
pot_df.insert(2, "POT", pot_df.pop("POT"))

# Pre-qualified Big Five + host country are placed in Pot00
pot_df.loc[pot_df["FullCountryName"].isin(LIST_POT_00_PREQUALIFIED), "POT"] = "Pot00"

pot_df

,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia
0,Belgium,BE,Pot02,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,91422,0,74529,60685,97257,12001
1,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790
2,Estonia,EE,Pot03,AGGREGATED,61921,103959,109643,95444,42056,55468,34756,38263,0,90089,37495,114399,23581,81317,26804
3,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239
4,Georgia,GE,Pot02,AGGREGATED,22623,75601,46846,9905,3750,0,23166,24521,65585,49739,28214,28763,75289,59033,6976
5,Greece,GR,Pot03,AGGREGATED,84356,16916,16659,0,93256,36604,79944,26493,43248,81486,98868,33928,73547,57259,20203
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019
7,Israel,IL,Pot02,AGGREGATED,14344,12439,72592,64917,44282,89143,25968,28388,37926,0,37495,114399,23581,81317,26804
8,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456
9,Lithuania,LT,Pot03,AGGREGATED,8954,3643,18035,8169,16090,16502,8592,14772,15620,10714,15556,0,3771,571,16555


---

## Stage 3: Pot threshold check

For each voting country, compute the total number of audience votes sent (`TOTAL_SENT_VOTES`) and check whether it meets the minimum threshold (`POT_VOTE_COUNT_THRESHOLD`).

- If **all** countries are above the threshold, no pot replacement is required and the workbook can jump straight to the jury voting stage.
- If **at least one** country falls below the threshold, the next stage will recompute its votes from the pot.

### Total sent votes & threshold flag

For each country, sum every `nVotesforDDI*` column to get the total number of audience votes sent and compare it to the threshold.


In [11]:
# Sum total audience votes sent by each country
votes_columns = [col for col in pot_df.columns if col.startswith("nVotesforDDI")]
pot_df["TOTAL_SENT_VOTES"] = pot_df[votes_columns].sum(axis=1)

# Flag countries that meet the minimum threshold
pot_df["ABOVE_POT_THRESHOLD"] = pot_df["TOTAL_SENT_VOTES"] >= POT_VOTE_COUNT_THRESHOLD

# Decide whether the pot-replacement stage is required at all
if False in pot_df["ABOVE_POT_THRESHOLD"].values:
    required_pot_calculation = True
    print("At least one country does not meet the vote threshold value, triggering pot calculations...")
    threshold_df = pot_df
else:
    required_pot_calculation = False
    print("All countries above threshold vote limit, no pot calculations required.")
    print('Go to cell containing "### FIRST STEP OF JURY VOTING ###"')

# Sort and persist the pre-pot snapshot
pot_df = pot_df.sort_values(["POT", "TOTAL_SENT_VOTES", "FullCountryName"])
pot_df.to_excel(
    os.path.join(SCRIPT_XLSX_DIRECTORY, "01.prePot_Calculations_voting_matrix.xlsx"),
    index=False,
)

pot_df

All countries above threshold vote limit, no pot calculations required.
Go to cell containing "### FIRST STEP OF JURY VOTING ###"


,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
1,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True
8,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True
15,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True
14,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True
16,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True
3,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True
11,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True
12,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True
10,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True


---

## Stage 3a: Pot replacement calculation

For each country below the threshold:

1. `get_pots(...)` decides whether to use **audience** votes from the same pot (and Pot00) or fall back to **jury** votes.
2. `calcul_replacement(...)` and `calcul_replacement_no_votes(...)` compute the equalised pot mean that will replace the country's audience vote breakdown.
3. The result is written back into `threshold_df` and exported to per-country Excel files.

### Helper: `get_pots`

Decide whether a country falls back to **audience** or **jury** votes, and return the dataframe of donor countries used to recompute the pot mean.


In [12]:
def get_pots(country_name, df_all_countries):
    """For a country below the threshold, decide which votes to use as replacement.

    Returns a tuple ``(mode, df_used)`` where ``mode`` is either ``'audience'``
    or ``'jury'`` and ``df_used`` is the dataframe of donor countries used to
    recompute the equalised pot mean (empty string when ``mode == 'jury'``).
    """
    pot = df_all_countries.loc[
        df_all_countries["FullCountryName"] == country_name, "POT"
    ].unique()[0]

    same_pot = df_all_countries[df_all_countries["POT"] == pot]
    pot_0 = df_all_countries[df_all_countries["POT"] == "Pot00"]

    same_pot_above_t = same_pot.loc[same_pot["ABOVE_POT_THRESHOLD"] == True]
    pot_0_above_t = pot_0.loc[pot_0["ABOVE_POT_THRESHOLD"] == True]

    # Pot00 (prequalified countries) follows its own rules
    if pot == "Pot00":
        if len(pot_0_above_t) > 1:
            return "audience", pot_0_above_t
        return "jury", ""

    # Standard SF pots
    if len(same_pot_above_t) >= 2:
        return "audience", same_pot_above_t
    if (len(same_pot_above_t) == 1) and (len(pot_0_above_t) > 0):
        return "audience", pd.concat([same_pot_above_t, pot_0_above_t])
    if (len(same_pot_above_t) == 1) and (len(pot_0_above_t) == 0):
        return "jury", ""
    if (len(same_pot_above_t) == 0) and (len(pot_0_above_t) > 1):
        return "audience", pd.concat([same_pot_above_t, pot_0_above_t])
    if (len(same_pot_above_t) == 0) and (len(pot_0_above_t) <= 1):
        return "jury", ""

    print("Something weird happened")
    return "", ""

### Helper: `calcul_replacement`

Equalised-pot calculation when the country **did** send some votes.


In [13]:
def calcul_replacement(country, tmp_df_pot,points_columns,country_name ):
    print('Calculating pot votes replacement for ', country_name)
    
    # Calculate total votes and first factor
    tmp_df_pot.insert(0,"total_votes",tmp_df_pot[points_columns].sum(axis=1))
    tmp_df_pot.insert(0,"factor",tmp_df_pot["total_votes"] / int(tmp_row_country[points_columns].sum(axis=1)))
    
    # Calculate equlized votes
    tmp_df_pot2 = tmp_df_pot[points_columns].div(tmp_df_pot["factor"], axis=0)
    tmp_df_pot2 = pd.DataFrame(tmp_df_pot2, columns=points_columns, index=tmp_df_pot.index)
    tmp_df_pot2['FullCountryName'] = tmp_df_pot['FullCountryName']
    
    # Prepare new dataframe 
    mean_values = tmp_df_pot2.drop('FullCountryName', axis=1).mean()
    new_row_data = mean_values.to_dict()
    new_row_data['FullCountryName'] = 'mean'
    tmp_df_pot3 = pd.concat([tmp_df_pot2,pd.DataFrame([new_row_data])], ignore_index=True)
    
    # Calculate mask and means 
    means = {}
    for col in points_columns:
        country_suffix = col.replace('nVotesforDDI', '').split('_')[-1]
        mask = ~tmp_df_pot2['FullCountryName'].str.endswith(country_suffix)
        means[col] = tmp_df_pot2.loc[mask, col].mean()
        mean_index = tmp_df_pot3[tmp_df_pot3['FullCountryName'] == 'mean'].index
        if country_suffix.replace(" ", "") == country_name.replace(" ", ""):
            tmp_df_pot3.loc[mean_index,col] = 0
        else :
            tmp_df_pot3.loc[mean_index,col] = means[col]

    # Calculate factor 2, and apply it to all votes
    factor2 = (POT_VOTE_COUNT_THRESHOLD-int(tmp_row_country[points_columns].sum(axis=1)))/ tmp_df_pot3.loc[mean_index,points_columns].sum(axis=1)
    new_mean = tmp_df_pot3.loc[mean_index,points_columns] * factor2.values[0]
    new_mean.insert(len(new_mean.columns), "FullCountryName","norm_mean")
    tmp_df_pot3 = pd.concat([tmp_df_pot3,new_mean])
    
    # Save pot calculation to Excel
    name_excel = 'pot_calculation_for_' + str(country_name) + '.xlsx'
    tmp_df_pot3.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, name_excel
    ), index=False)
    
    return tmp_df_pot3

### Helper: `calcul_replacement_no_votes`

Equalised-pot calculation when the country sent **no** votes at all.


In [14]:
def calcul_replacement_no_votes(country, tmp_df_pot,points_columns,country_name):
    print('Calculating pot votes replacement for ', country_name, ' with no votes delivered process.')
    
     # Calculate total votes and first factor
    tmp_df_pot.insert(0,"total_votes",tmp_df_pot[points_columns].sum(axis=1))
    tmp_df_pot.insert(0,"factor",tmp_df_pot[points_columns].sum(axis=1))
    
    # Calculate equlized votes
    tmp_df_pot2 = tmp_df_pot[points_columns].div(tmp_df_pot["factor"], axis=0)
    tmp_df_pot2 = pd.DataFrame(tmp_df_pot2, columns=points_columns, index=tmp_df_pot.index)
    tmp_df_pot2['FullCountryName'] = tmp_df_pot['FullCountryName']
    
     # Prepare new dataframe 
    mean_values = tmp_df_pot2.drop('FullCountryName', axis=1).mean()
    new_row_data = mean_values.to_dict()
    new_row_data['FullCountryName'] = 'mean'
    tmp_df_pot3 = pd.concat([tmp_df_pot2,pd.DataFrame([new_row_data])], ignore_index=True)
    
     # Calculate mask and means 
    means = {}
    for col in points_columns:
        country_suffix = col.replace('nVotesforDDI', '').split('_')[-1]
        mask = ~tmp_df_pot2['FullCountryName'].str.endswith(country_suffix)
        means[col] = tmp_df_pot2.loc[mask, col].mean()
        mean_index = tmp_df_pot3[tmp_df_pot3['FullCountryName'] == 'mean'].index
        if country_suffix.replace(" ", "") == country_name.replace(" ", ""):
            tmp_df_pot3.loc[mean_index,col] = 0
        else :
            tmp_df_pot3.loc[mean_index,col] = means[col]
            
    # Calculate factor 2, and apply it to all votes
    factor2 = POT_VOTE_COUNT_THRESHOLD
    new_mean = tmp_df_pot3.loc[mean_index,points_columns] * factor2
    new_mean.insert(len(new_mean.columns), "FullCountryName","norm_mean")
    tmp_df_pot3 = pd.concat([tmp_df_pot3,new_mean])
    
    # Save pot calculation to Excel
    name_excel = 'pot_calculation_for_' + str(country_name) + '.xlsx'
    tmp_df_pot3.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, name_excel
    ), index=False)
    
    return tmp_df_pot3

### Apply pot replacement to all countries below threshold


In [15]:
threshold_df = pot_df

if required_pot_calculation:
    
    list_under_countries = threshold_df.loc[threshold_df["ABOVE_POT_THRESHOLD"] == False,"FullCountryName"].unique()
    points_columns = [col for col in threshold_df.columns if col.startswith('nVotesfor')]
    
    # Deal with all countries below threshold
    for country in list_under_countries:
        
        # Take all case into account, say if we want pot calculation or jury voting. Give voting to use
        mode_, df_used = get_pots(country, threshold_df)
        print(country," : " ,mode_)
        
        
        # San Marino (PDF §1.6): no audience vote is conducted, so always use the pot result
        # in semi-finals regardless of what get_pots() returned.
        if (country.replace(" ", "") == 'SanMarino') and (selected_df_name != 'gf_televoting_df'):
            print(f"San Marino — forcing audience-pot calculation per PDF §1.6.")
            if mode_ != 'audience':
                # If get_pots() decided 'jury' (e.g. not enough valid pot members), still
                # build a pot-substitute using whatever donors are available.
                same_pot = threshold_df[threshold_df["POT"] == threshold_df.loc[
                    threshold_df["FullCountryName"] == country, "POT"
                ].unique()[0]]
                pot_0 = threshold_df[threshold_df["POT"] == "Pot00"]
                df_used = pd.concat([
                    same_pot[same_pot["ABOVE_POT_THRESHOLD"] == True],
                    pot_0[pot_0["ABOVE_POT_THRESHOLD"] == True],
                ])
                mode_ = 'audience'

        # Pot calculation with audiences voting of same pot
        if mode_ == 'audience':
            
            if country.replace(" ", "") == 'SanMarino':
                print("Checking San Marino - Audience")
                for col in points_columns:
                    if col != "nVotesforDDI13_San Marino":
                        threshold_df.loc[threshold_df["FullCountryName"] == country,col] = 1
            
            tmp_row_country = threshold_df.loc[threshold_df["FullCountryName"] == country]
            
            if threshold_df.loc[threshold_df["FullCountryName"] == country, "TOTAL_SENT_VOTES"].unique()[0] > 0:
                df = calcul_replacement(tmp_row_country, df_used,points_columns,country)
            else :
                df = calcul_replacement_no_votes(tmp_row_country, df_used,points_columns,country)
            
            #print(country)
            #display(df)
            
            for col in points_columns:
                threshold_df.loc[threshold_df["FullCountryName"] == country,col] = round(df.loc[df["FullCountryName"] == 'norm_mean',col].unique()[0]) + threshold_df.loc[threshold_df["FullCountryName"] == country,col].unique()[0]
        
        # If audience vote need to be replaced by jury votes
        else :
            print('Votes audience of ', country, ' should be replaced by jury votes of this country.')
            list_replace_audience_by_jury.append(country)
            for col in points_columns:
                threshold_df.loc[threshold_df["FullCountryName"] == country,col] = 0
                
threshold_df["TOTAL_SENT_VOTES"] = threshold_df[votes_columns].sum(axis=1)
threshold_df

,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
1,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True
8,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True
15,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True
14,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True
16,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True
3,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True
11,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True
12,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True
10,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True


---

---

## Stage 4: Jury calculation

Cleaning up the jury dataframe so its columns match the audience matrix (DDI number suffixed with the recipient country name).

In [16]:
############# JURY VOTING 1 #############

# Generating corresponding jury pts 
if selected_df_name == 'sf1_televoting_df':
    jury_df = sf1_jury_df
elif selected_df_name == 'sf2_televoting_df':
    jury_df = sf2_jury_df
elif selected_df_name == 'gf_televoting_df':
    jury_df = gf_jury_df
else:
    raise Exception("Unexpected televoting dataframe name, can't find matching jury dataframe.")

# Append full country names to nVotesforDDIxx columns names
for ddi_key, country_name in corresponding_ddi_dict.items():
    ddi_number = ddi_key[-2:]
    column_to_update = [col for col in jury_df.columns if col.startswith(f"nPointsforDDI{ddi_number}")][0]
    new_column_name = f"Rankfor{ddi_number}_{country_name}"
    jury_df = jury_df.rename(columns={column_to_update: new_column_name})

jury_df

,strName,CountryId,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,nlsReplacementResultFromPotCountries
0,Moldova,39,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,1
1,Sweden,32,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,0
2,Croatia,7,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,0
3,Greece,15,3,7,10,0,8,2,6,14,1,13,4,9,12,11,5,0
4,Portugal,26,9,6,10,2,0,14,5,1,7,12,8,13,11,3,4,0
5,Georgia,42,6,8,9,3,5,0,4,2,12,10,7,1,14,11,13,0
6,Finland,12,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,0
7,Montenegro,43,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,0
8,Estonia,10,8,11,2,10,1,5,3,7,0,12,14,13,9,6,4,0
9,Israel,18,6,5,13,9,3,11,8,4,14,0,10,1,2,12,7,0


---

---

### Stage 4a: Convert jury ranks to Eurovision points

Apply the official mapping (`1 -> 12, 2 -> 10, 3 -> 8, ..., 10 -> 1`) to every `Rankfor##_<Country>` column, producing the corresponding `nPointsforDDI##_<Country>` columns.

In [17]:
############# JURY VOTING 2 #############
# Convert each jury rank into Eurovision points using the official mapping.

jury_pts_df = jury_df


def points_mapping(rank):
    """Map a jury rank (1..10) to Eurovision points; everything else => 0."""
    mapping = {1: 12, 2: 10, 3: 8, 4: 7, 5: 6, 6: 5, 7: 4, 8: 3, 9: 2, 10: 1}
    return mapping.get(rank, 0)


# Add an nPointsforDDI##_<Country> column derived from each Rankfor##_<Country> column
for column in jury_pts_df.columns:
    if column.startswith("Rankfor"):
        new_column_name = column.replace("Rankfor", "nPointsforDDI")
        jury_pts_df[new_column_name] = jury_pts_df[column].apply(points_mapping)

jury_pts_df

,strName,CountryId,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,nlsReplacementResultFromPotCountries,nPointsforDDI01_Moldova,nPointsforDDI02_Sweden,nPointsforDDI03_Croatia,nPointsforDDI04_Greece,nPointsforDDI05_Portugal,nPointsforDDI06_Georgia,nPointsforDDI07_Finland,nPointsforDDI08_Montenegro,nPointsforDDI09_Estonia,nPointsforDDI10_Israel,nPointsforDDI11_Belgium,nPointsforDDI12_Lithuania,nPointsforDDI13_San Marino,nPointsforDDI14_Poland,nPointsforDDI15_Serbia
0,Moldova,39,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,1,0,1,3,4,6,0,5,7,0,0,8,12,10,0,2
1,Sweden,32,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,0,3,0,8,0,0,7,6,1,12,0,10,5,2,0,4
2,Croatia,7,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,0,8,5,0,6,7,0,0,0,10,1,3,4,0,2,12
3,Greece,15,3,7,10,0,8,2,6,14,1,13,4,9,12,11,5,0,8,4,1,0,3,10,5,0,12,0,7,2,0,0,6
4,Portugal,26,9,6,10,2,0,14,5,1,7,12,8,13,11,3,4,0,2,5,1,10,0,0,6,12,4,0,3,0,0,8,7
5,Georgia,42,6,8,9,3,5,0,4,2,12,10,7,1,14,11,13,0,5,3,2,8,6,0,7,10,0,1,4,12,0,0,0
6,Finland,12,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,0,1,5,0,8,6,2,0,0,3,7,12,4,10,0,0
7,Montenegro,43,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,0,6,1,0,8,5,0,0,0,2,7,12,0,3,10,4
8,Estonia,10,8,11,2,10,1,5,3,7,0,12,14,13,9,6,4,0,3,0,10,1,12,6,8,4,0,0,0,0,2,5,7
9,Israel,18,6,5,13,9,3,11,8,4,14,0,10,1,2,12,7,0,5,6,0,2,8,0,3,7,0,0,1,12,10,0,4


---

---

### Stage 4b: Final-jury tie-break (PDF §1.3.1)

Resolve ties on **total jury points** for each recipient country. Per the PDF (§1.3.1, "Tie due to the same number of points from all National Juries"), the chain is:

1. Highest number of national juries that gave it any points.
2. Highest number of `12`-point scores; then `10`, `8`, `7`, `6`, `5`, `4`, `3`, `2`, `1`.
3. Earlier in the show running order wins (replaces the previous alphabetical fallback).

Note: this is **not** the within-jury tie-breaker (majority/youngest/show-of-hands, PDF §1.3.1 second box) — that one is applied upstream by the jury-web interface and is already baked into the per-jury ranks in the input CSV.

In [18]:
############# JURY VOTING 3 : Tie breaking #############

# Step 1: Filter the columns and transpose the filtered DataFrame
filtered_columns = [col for col in jury_pts_df.columns if col.startswith("nPointsforDDI")]
filtered_df = jury_pts_df[filtered_columns]
transposed_df = filtered_df.T

# Step 3: Rename the columns
jury_pts_df = jury_pts_df.rename(columns={"Unnamed: 0": "strName"})
transposed_df.columns = jury_pts_df["strName"].values
# RoW is the Pot 0 jury-pot result, not a real national jury — drop it from the per-country jury aggregation
# (case-insensitive match to handle 'Rest of World' vs 'Rest Of World').
row_cols = [c for c in transposed_df.columns if str(c).lower().replace(' ','') == 'restofworld']
if row_cols:
    transposed_df.drop(columns=row_cols, inplace=True)
jury_tiebreaking_df = transposed_df
country_columns = jury_tiebreaking_df.columns
jury_tiebreaking_df = jury_tiebreaking_df.reset_index()
jury_tiebreaking_df = jury_tiebreaking_df.rename(columns={"index": "Recipient_Country"})


jury_tiebreaking_df["Pts_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row != 0).sum(), axis=1)
jury_tiebreaking_df["12_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 12).sum(), axis=1)
jury_tiebreaking_df["10_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 10).sum(), axis=1)
jury_tiebreaking_df["8_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 8).sum(), axis=1)
jury_tiebreaking_df["7_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 7).sum(), axis=1)
jury_tiebreaking_df["6_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 6).sum(), axis=1)
jury_tiebreaking_df["5_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 5).sum(), axis=1)
jury_tiebreaking_df["4_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 4).sum(), axis=1)
jury_tiebreaking_df["3_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 3).sum(), axis=1)
jury_tiebreaking_df["2_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 2).sum(), axis=1)
jury_tiebreaking_df["1_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 1).sum(), axis=1)

# Sort in tiebreaking order
# Final tie-break per EBU PDF §1.3.1: jury count → 12-pt count → 10-pt count → … → 1-pt count
# → show running order (earlier wins). The DDI prefix in 'Recipient_Country' encodes the show order.
jury_tiebreaking_df["_ShowOrder"] = (
    jury_tiebreaking_df["Recipient_Country"].str.extract(r"DDI(\d+)")[0].astype(int)
)
jury_tiebreaking_df = jury_tiebreaking_df.sort_values(
    by=[
        "Pts_from_X_National_Juries",
        "12_from_X_National_Juries",
        "10_from_X_National_Juries",
        "8_from_X_National_Juries",
        "7_from_X_National_Juries",
        "6_from_X_National_Juries",
        "5_from_X_National_Juries",
        "4_from_X_National_Juries",
        "3_from_X_National_Juries",
        "2_from_X_National_Juries",
        "1_from_X_National_Juries",
        "_ShowOrder",
    ],
    ascending=[False, False, False, False, False, False, False, False, False, False, False, True],
)
jury_tiebreaking_df = jury_tiebreaking_df.drop(columns=["_ShowOrder"])

columns = [
    "Pts_from_X_National_Juries",
    "12_from_X_National_Juries",
    "10_from_X_National_Juries",
    "8_from_X_National_Juries",
    "7_from_X_National_Juries",
    "6_from_X_National_Juries",
    "5_from_X_National_Juries",
    "4_from_X_National_Juries",
    "3_from_X_National_Juries",
    "2_from_X_National_Juries",
    "1_from_X_National_Juries",
]

# Move the specified columns right after the first column
cols = jury_tiebreaking_df.columns.tolist()
for col in reversed(columns):
    if col in cols:
        cols.insert(1, cols.pop(cols.index(col)))
jury_tiebreaking_df = jury_tiebreaking_df[cols]

# Add a new column 'TIEBREAKING_RANK'
jury_tiebreaking_df.insert(1, "TIEBREAKING_RANK", range(1, len(jury_tiebreaking_df) + 1))

# Save the new dataframe
jury_tiebreaking_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "03.jury_received_pts_tiebreakers.xlsx"), index=False)

jury_tiebreaking_df

,Recipient_Country,TIEBREAKING_RANK,Pts_from_X_National_Juries,12_from_X_National_Juries,10_from_X_National_Juries,8_from_X_National_Juries,7_from_X_National_Juries,6_from_X_National_Juries,5_from_X_National_Juries,4_from_X_National_Juries,3_from_X_National_Juries,2_from_X_National_Juries,1_from_X_National_Juries,Moldova,Sweden,Croatia,Greece,Portugal,Georgia,Finland,Montenegro,Estonia,Israel,Belgium,Lithuania,San Marino,Poland,Serbia,Germany,Italy
10,nPointsforDDI11_Belgium,1,14,2,2,2,1,0,1,1,3,0,2,8,10,3,7,3,4,12,12,0,1,0,0,5,10,1,8,3
3,nPointsforDDI04_Greece,2,14,0,1,4,1,3,0,2,0,1,2,4,0,6,0,10,8,8,8,1,2,0,8,1,7,6,4,6
8,nPointsforDDI09_Estonia,3,13,3,1,1,0,0,0,3,2,3,0,0,12,10,12,4,0,3,2,0,0,4,4,2,2,3,12,8
14,nPointsforDDI15_Serbia,4,13,2,1,0,3,2,0,3,0,1,1,2,4,12,6,7,0,0,4,7,4,10,12,6,0,0,7,1
0,nPointsforDDI01_Moldova,5,13,1,1,3,1,1,2,0,2,1,1,0,3,8,8,2,5,1,6,3,5,7,10,8,0,12,0,0
4,nPointsforDDI05_Portugal,6,13,1,0,2,1,5,1,0,2,1,0,6,0,7,3,0,6,6,5,12,8,0,2,3,6,8,6,0
1,nPointsforDDI02_Sweden,7,13,0,0,0,1,1,5,1,2,1,2,1,0,5,4,5,3,5,1,0,6,0,0,7,3,5,2,5
6,nPointsforDDI07_Finland,8,12,1,0,1,1,3,3,0,2,0,1,5,6,0,5,6,7,0,0,8,3,6,3,12,5,0,1,0
2,nPointsforDDI03_Croatia,9,12,0,3,2,0,0,1,1,2,1,2,3,8,0,1,1,2,0,0,10,0,5,0,4,8,10,3,10
7,nPointsforDDI08_Montenegro,10,11,1,2,0,3,1,0,2,0,0,2,7,1,0,0,12,10,0,0,4,7,1,6,0,0,7,10,4


---

---

### Stage 4c: Jury total points

Add two summary rows to the jury matrix:

- `Total_with_RoW` - sum of jury points received from every national jury (incl. Rest Of World).
- `Total_without_RoW` - same total, but excluding Rest Of World.

Also add `nDifferenceforDDI*` columns derived from the rank, used later as a tie-break helper inside the televoting stage.

In [19]:
############# JURY VOTING 3 : Total points #############
# Add Total_with_RoW and Total_without_RoW summary rows, plus the nDifferenceforDDI*
# helper columns used later as a tie-breaker on the televoting side.

jury_totals_df = jury_pts_df
pts_columns = [col for col in jury_totals_df.columns if col.startswith("nPointsforDDI")]

# --- Row: Total_with_RoW (sum of all national juries, including Rest Of World) ---
new_row_sums_jury = pd.DataFrame(columns=jury_totals_df.columns, index=[len(jury_totals_df)])
for col in pts_columns:
    new_row_sums_jury[col] = jury_totals_df[col].sum()
new_row_sums_jury["strName"] = "Total_with_RoW"
jury_totals_df = pd.concat([jury_totals_df, new_row_sums_jury], ignore_index=True)

# --- Row: Total_without_RoW (Total_with_RoW minus the Rest Of World contribution) ---
new_row_without_RoW = pd.DataFrame(columns=jury_totals_df.columns, index=[len(jury_totals_df)])
new_row_without_RoW["strName"] = "Total_without_RoW"
# Match RoW case-insensitively (CSV may use "Rest of World" or "Rest Of World")
row_RoW = jury_totals_df[jury_totals_df["strName"].str.lower().str.replace(" ","") == "restofworld"]

for col in pts_columns:
    column_sum_without_RoW = (
        jury_totals_df.loc[jury_totals_df["strName"] == "Total_with_RoW", col].values[0]
        - row_RoW[col].values[0]
    )
    new_row_without_RoW[col] = column_sum_without_RoW

jury_totals_df = pd.concat([jury_totals_df, new_row_without_RoW], ignore_index=True)

# Note: removed the nDifferenceforDDI* helper columns (the "1 - rank/10000" perturbation hack).
# The audience tie-break per PDF §1.2.1 — "song with best rank from the National Jury wins" —
# is now applied cleanly in the televoting cells using the actual jury rank as the tie-break key.

jury_totals_df

,strName,CountryId,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,nlsReplacementResultFromPotCountries,nPointsforDDI01_Moldova,nPointsforDDI02_Sweden,nPointsforDDI03_Croatia,nPointsforDDI04_Greece,nPointsforDDI05_Portugal,nPointsforDDI06_Georgia,nPointsforDDI07_Finland,nPointsforDDI08_Montenegro,nPointsforDDI09_Estonia,nPointsforDDI10_Israel,nPointsforDDI11_Belgium,nPointsforDDI12_Lithuania,nPointsforDDI13_San Marino,nPointsforDDI14_Poland,nPointsforDDI15_Serbia
0,Moldova,39,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,1,0,1,3,4,6,0,5,7,0,0,8,12,10,0,2
1,Sweden,32,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,0,3,0,8,0,0,7,6,1,12,0,10,5,2,0,4
2,Croatia,7,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,0,8,5,0,6,7,0,0,0,10,1,3,4,0,2,12
3,Greece,15,3,7,10,0,8,2,6,14,1,13,4,9,12,11,5,0,8,4,1,0,3,10,5,0,12,0,7,2,0,0,6
4,Portugal,26,9,6,10,2,0,14,5,1,7,12,8,13,11,3,4,0,2,5,1,10,0,0,6,12,4,0,3,0,0,8,7
5,Georgia,42,6,8,9,3,5,0,4,2,12,10,7,1,14,11,13,0,5,3,2,8,6,0,7,10,0,1,4,12,0,0,0
6,Finland,12,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,0,1,5,0,8,6,2,0,0,3,7,12,4,10,0,0
7,Montenegro,43,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,0,6,1,0,8,5,0,0,0,2,7,12,0,3,10,4
8,Estonia,10,8,11,2,10,1,5,3,7,0,12,14,13,9,6,4,0,3,0,10,1,12,6,8,4,0,0,0,0,2,5,7
9,Israel,18,6,5,13,9,3,11,8,4,14,0,10,1,2,12,7,0,5,6,0,2,8,0,3,7,0,0,1,12,10,0,4


---

---

### Stage 4d: Jury global rankings

Rank every recipient country based on `Total_with_RoW` and `Total_without_RoW`, producing two new rows (`Rank_with_RoW`, `Rank_without_RoW`).

In [20]:
############# JURY VOTING 4 : CALCULATE RANK #############
# Add Rank_with_RoW and Rank_without_RoW rows derived from the totals rows above.

jury_global_rankings_df = jury_totals_df


def _build_rank_row(label, totals_label):
    """Return a one-row dataframe holding the per-country jury rank derived from ``totals_label``."""
    row = pd.DataFrame(columns=jury_global_rankings_df.columns, index=[len(jury_global_rankings_df)])
    row["strName"] = label
    totals = jury_global_rankings_df.loc[
        jury_global_rankings_df["strName"] == totals_label, pts_columns
    ]
    ranks = totals.rank(axis=1, ascending=False, method="min").astype(int)
    for i, col in enumerate(pts_columns):
        rank_col_name = col.replace("nPointsforDDI", "Rankfor")
        row[rank_col_name] = ranks.iloc[0, i]
    return row


jury_global_rankings_df = pd.concat(
    [jury_global_rankings_df, _build_rank_row("Rank_with_RoW", "Total_with_RoW")],
    ignore_index=True,
)
jury_global_rankings_df = pd.concat(
    [jury_global_rankings_df, _build_rank_row("Rank_without_RoW", "Total_without_RoW")],
    ignore_index=True,
)

jury_global_rankings_df

,strName,CountryId,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,nlsReplacementResultFromPotCountries,nPointsforDDI01_Moldova,nPointsforDDI02_Sweden,nPointsforDDI03_Croatia,nPointsforDDI04_Greece,nPointsforDDI05_Portugal,nPointsforDDI06_Georgia,nPointsforDDI07_Finland,nPointsforDDI08_Montenegro,nPointsforDDI09_Estonia,nPointsforDDI10_Israel,nPointsforDDI11_Belgium,nPointsforDDI12_Lithuania,nPointsforDDI13_San Marino,nPointsforDDI14_Poland,nPointsforDDI15_Serbia
0,Moldova,39,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,1,0,1,3,4,6,0,5,7,0,0,8,12,10,0,2
1,Sweden,32,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,0,3,0,8,0,0,7,6,1,12,0,10,5,2,0,4
2,Croatia,7,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,0,8,5,0,6,7,0,0,0,10,1,3,4,0,2,12
3,Greece,15,3,7,10,0,8,2,6,14,1,13,4,9,12,11,5,0,8,4,1,0,3,10,5,0,12,0,7,2,0,0,6
4,Portugal,26,9,6,10,2,0,14,5,1,7,12,8,13,11,3,4,0,2,5,1,10,0,0,6,12,4,0,3,0,0,8,7
5,Georgia,42,6,8,9,3,5,0,4,2,12,10,7,1,14,11,13,0,5,3,2,8,6,0,7,10,0,1,4,12,0,0,0
6,Finland,12,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,0,1,5,0,8,6,2,0,0,3,7,12,4,10,0,0
7,Montenegro,43,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,0,6,1,0,8,5,0,0,0,2,7,12,0,3,10,4
8,Estonia,10,8,11,2,10,1,5,3,7,0,12,14,13,9,6,4,0,3,0,10,1,12,6,8,4,0,0,0,0,2,5,7
9,Israel,18,6,5,13,9,3,11,8,4,14,0,10,1,2,12,7,0,5,6,0,2,8,0,3,7,0,0,1,12,10,0,4


---

---

### Stage 4e: Final jury results

Transpose the jury rankings so each row is a recipient country, attach the tie-breaking rank, sort using `(Total_without_RoW DESC, TIEBREAKING_RANK ASC)` and assign the final 1..N jury rank.

In [21]:
############# JURY FINAL RESULTS #############

# Build JURY final dataframe 
final_jury_results_df = jury_global_rankings_df.set_index('strName')
transposed_final_jury_results_df = final_jury_results_df.T
transposed_final_jury_results_df = transposed_final_jury_results_df.reset_index()
transposed_final_jury_results_df = transposed_final_jury_results_df.rename(columns={'index': 'Recipient_Country'})

# Filter rows that correspond to the 'nVotesforDDI' columns
transposed_final_jury_results_df = transposed_final_jury_results_df.sort_values(by='Total_without_RoW', ascending=False)

# Replace the RoW Jury with 0


# Move the 'jury_PTS_TOTAL' column to the second position
cols = transposed_final_jury_results_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('Total_with_RoW')))
cols.insert(1, cols.pop(cols.index('Total_without_RoW')))
transposed_final_jury_results_df = transposed_final_jury_results_df[cols]

transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_df
jury_tiebreaking_ranks_only_df = jury_tiebreaking_df[['Recipient_Country', 'TIEBREAKING_RANK']]
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df.merge(jury_tiebreaking_ranks_only_df, on='Recipient_Country', how='left')

# Move the 'jury_PTS_TOTAL' column to the second position
cols = transposed_final_jury_results_w_tiebreakers_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('TIEBREAKING_RANK')))
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df[cols]

# Sort for final order
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df.sort_values(by=['Total_without_RoW', 'TIEBREAKING_RANK'], ascending=[False, True])
transposed_final_jury_results_w_tiebreakers_df.insert(0, 'Final_Rank', range(1, len(transposed_final_jury_results_w_tiebreakers_df) + 1))

# save
transposed_final_jury_results_w_tiebreakers_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "04.jury_final_jury_results_w_tiebreaker.xlsx"), index=False)

transposed_final_jury_results_w_tiebreakers_df

,Final_Rank,Recipient_Country,Total_without_RoW,TIEBREAKING_RANK,Total_with_RoW,Moldova,Sweden,Croatia,Greece,Portugal,Georgia,Finland,Montenegro,Estonia,Israel,Belgium,Lithuania,San Marino,Poland,Serbia,Germany,Italy,Rest of World,Rank_with_RoW,Rank_without_RoW
0,1,nPointsforDDI11_Belgium,87,1.0,93,8,10,3,7,3,4,12,12,0,1,0,0,5,10,1,8,3,6,NaN,NaN
1,2,nPointsforDDI15_Serbia,82,4.0,86,2,4,12,6,7,0,0,4,7,4,10,12,6,0,0,7,1,4,NaN,NaN
2,3,nPointsforDDI04_Greece,79,2.0,84,4,0,6,0,10,8,8,8,1,2,0,8,1,7,6,4,6,5,NaN,NaN
5,4,nPointsforDDI09_Estonia,78,3.0,90,0,12,10,12,4,0,3,2,0,0,4,4,2,2,3,12,8,12,NaN,NaN
3,5,nPointsforDDI01_Moldova,78,5.0,78,0,3,8,8,2,5,1,6,3,5,7,10,8,0,12,0,0,0,NaN,NaN
4,6,nPointsforDDI05_Portugal,78,6.0,80,6,0,7,3,0,6,6,5,12,8,0,2,3,6,8,6,0,2,NaN,NaN
6,7,nPointsforDDI12_Lithuania,73,11.0,73,12,5,4,2,0,12,4,0,0,12,8,0,0,12,0,0,2,0,NaN,NaN
7,8,nPointsforDDI08_Montenegro,69,10.0,79,7,1,0,0,12,10,0,0,4,7,1,6,0,0,7,10,4,10,NaN,NaN
8,9,nPointsforDDI07_Finland,67,8.0,67,5,6,0,5,6,7,0,0,8,3,6,3,12,5,0,1,0,0,NaN,NaN
9,10,nPointsforDDI03_Croatia,65,9.0,72,3,8,0,1,1,2,0,0,10,0,5,0,4,8,10,3,10,7,NaN,NaN


---

## Stage 4f: Replace audience voting by jury for selected countries

For any country flagged in `list_replace_audience_by_jury` (typically San Marino, or any country that fell back to jury during pot calculation), copy the jury points into the audience matrix so the final televoting calculation uses the jury values instead.

In [22]:
def replace_audience_w_jury(country, df_jury, df_audience):
    pts_columns_audiences = [col for col in df_audience.columns if col.startswith("nVotesforDDI")]
    pts_columns_jury = [col for col in df_jury.columns if col.startswith("nPointsforDDI")]
    for col_audience in pts_columns_audiences:
        for col_jury in pts_columns_jury:
            if col_jury.split('_')[1] == col_audience.split('_')[1]:
                df_audience.loc[df_audience["FullCountryName"]==country, col_audience] = df_jury.loc[df_jury["strName"]==country, col_jury].unique()[0]
    return df_audience

### Inspect the list of countries to replace with jury votes


In [23]:
#list_replace_audience_by_jury.append('San Marino')
list_replace_audience_by_jury

[]

### Apply jury replacement on the audience matrix


In [24]:
print('Replacing audience voting by jury voting for : ' , list_replace_audience_by_jury )
for country in list_replace_audience_by_jury:
    threshold_df = replace_audience_w_jury(country, jury_global_rankings_df, threshold_df)
threshold_df

Replacing audience voting by jury voting for :  []


,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
1,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True
8,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True
15,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True
14,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True
16,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True
3,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True
11,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True
12,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True
10,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True


---

## Stage 5: Televoting calculation

Merge the audience matrix with the jury `nDifferenceforDDI*` helper columns. Those small (< 1) values will be added to the audience votes as a deterministic tie-breaker before ranking.

In [25]:
############# TELEVOTING 1 #############
# Per PDF §1.2.1: when two songs receive the same number of audience votes in a given
# country, the tie is broken by that country's own National Jury rank (lower = better).
# We attach each voting country's jury Rankfor* columns to its televoting row so we can
# break ties cleanly when ranking audience votes below.

if required_pot_calculation:
    televoting_final_votes_w_jury_pts_df = threshold_df
else:
    televoting_final_votes_w_jury_pts_df = pot_df

# Pull each country's own jury final ranks as tie-break keys
jury_rank_cols = [c for c in jury_global_rankings_df.columns if c.startswith("Rankfor")]
jury_ranks_for_tiebreak = jury_global_rankings_df[["strName"] + jury_rank_cols].copy()

televoting_final_votes_w_jury_pts_df = televoting_final_votes_w_jury_pts_df.merge(
    jury_ranks_for_tiebreak, left_on="FullCountryName", right_on="strName", how="left"
)

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia
0,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True,Germany,12,9,8,7,5,15,10,2,1,6,3,11,14,13,4
1,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True,Italy,13,6,2,5,11,14,12,7,3,15,8,9,4,1,10
2,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True,Serbia,1,6,2,5,3,7,12,4,8,9,10,14,11,13,0
4,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True,Sweden,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7
5,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True,Finland,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True,Croatia,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1
7,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True,Montenegro,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7
8,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True,Poland,12,8,3,4,5,10,6,11,9,7,2,1,13,0,14
9,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True,Moldova,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9


---

### Stage 5b: Audience tie-breaking

Avoid ties in the audience ranking by adding the jury-derived difference (`nDifferenceforDDI* / 100`) to each `nVotesforDDI*` column before ranking.

### Quick inspection of the merged televoting + jury-difference matrix


In [26]:
televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia
0,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True,Germany,12,9,8,7,5,15,10,2,1,6,3,11,14,13,4
1,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True,Italy,13,6,2,5,11,14,12,7,3,15,8,9,4,1,10
2,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True,Serbia,1,6,2,5,3,7,12,4,8,9,10,14,11,13,0
4,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True,Sweden,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7
5,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True,Finland,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True,Croatia,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1
7,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True,Montenegro,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7
8,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True,Poland,12,8,3,4,5,10,6,11,9,7,2,1,13,0,14
9,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True,Moldova,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9


### Stage 5c: Apply jury-derived tie-breaker to the audience votes


In [27]:
############# TELEVOTING 2 #############
# Previously this cell perturbed audience vote counts by tiny fractions derived from the
# jury rank (the nDifferenceforDDI hack). That implicit tie-break is replaced by an
# explicit ranking step in TELEVOTING 3 that uses the jury rank as a secondary sort key.
# This cell is now a no-op kept for compatibility with downstream cell numbering.

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia
0,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True,Germany,12,9,8,7,5,15,10,2,1,6,3,11,14,13,4
1,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True,Italy,13,6,2,5,11,14,12,7,3,15,8,9,4,1,10
2,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True,Serbia,1,6,2,5,3,7,12,4,8,9,10,14,11,13,0
4,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True,Sweden,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7
5,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True,Finland,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True,Croatia,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1
7,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True,Montenegro,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7
8,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True,Poland,12,8,3,4,5,10,6,11,9,7,2,1,13,0,14
9,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True,Moldova,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9


### Stage 5d: Rank audience votes per voting country


In [28]:
############# TELEVOTING 3 #############
# Rank audience votes per voting country, breaking ties by the country's own jury rank
# (lower jury rank = better) per PDF §1.2.1.

import numpy as np

votes_columns = [col for col in televoting_final_votes_w_jury_pts_df.columns if col.startswith('nVotesforDDI')]
rank_columns  = [col.replace('nVotesforDDI', 'RankforDDI') for col in votes_columns]

def _rank_with_jury_tiebreak(row):
    # For each DDI column: (-votes, jury_rank). lexsort gives small-first so negating votes
    # makes higher-vote = better. jury_rank is naturally smaller-is-better.
    keys = []
    for v_col in votes_columns:
        ddi = v_col.replace('nVotesforDDI', '')      # e.g. '01_Moldova'
        jury_col = 'Rankfor' + ddi                   # matches the merged column name
        votes = row[v_col]
        jury_r = row.get(jury_col, np.nan)
        if pd.isna(jury_r) or jury_r == 0:
            jury_r = 99   # disqualified / home / unknown — push to the back of any tie
        keys.append((-votes, jury_r))
    # Compute final 1..N order
    order = sorted(range(len(keys)), key=lambda i: keys[i])
    ranks = [0] * len(keys)
    for pos, i in enumerate(order):
        ranks[i] = pos + 1
    return pd.Series(ranks, index=rank_columns)

televoting_final_votes_w_jury_pts_df[rank_columns] = (
    televoting_final_votes_w_jury_pts_df.apply(_rank_with_jury_tiebreak, axis=1)
)

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,RankforDDI01_Moldova,RankforDDI02_Sweden,RankforDDI03_Croatia,RankforDDI04_Greece,RankforDDI05_Portugal,RankforDDI06_Georgia,RankforDDI07_Finland,RankforDDI08_Montenegro,RankforDDI09_Estonia,RankforDDI10_Israel,RankforDDI11_Belgium,RankforDDI12_Lithuania,RankforDDI13_San Marino,RankforDDI14_Poland,RankforDDI15_Serbia
0,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True,Germany,12,9,8,7,5,15,10,2,1,6,3,11,14,13,4,7,3,8,15,12,6,9,2,11,5,14,13,10,1,4
1,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True,Italy,13,6,2,5,11,14,12,7,3,15,8,9,4,1,10,9,1,2,14,13,11,5,12,6,10,8,7,15,4,3
2,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,3,13,4,7,5,9,1,6,8,12,15,10,11,14
3,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True,Serbia,1,6,2,5,3,7,12,4,8,9,10,14,11,13,0,13,10,4,8,1,5,14,11,12,6,7,9,3,2,15
4,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True,Sweden,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,4,15,2,1,14,7,10,5,12,11,6,3,9,13,8
5,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True,Finland,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,12,14,11,7,6,1,15,5,4,8,13,10,9,2,3
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True,Croatia,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,14,10,15,12,11,13,6,1,3,4,7,2,9,5,8
7,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True,Montenegro,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,14,12,6,1,5,8,11,15,7,3,4,2,13,9,10
8,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True,Poland,12,8,3,4,5,10,6,11,9,7,2,1,13,0,14,8,3,10,9,5,12,7,2,1,4,13,11,6,15,14
9,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True,Moldova,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,15,14,11,6,9,10,4,13,3,1,5,8,7,2,12


### Stage 5e: Map audience ranks to Eurovision points


In [29]:
############# TELEVOTING 4 #############

# Change ranks to points
televoting_final_pts_df = televoting_final_votes_w_jury_pts_df
rank_columns = [col for col in televoting_final_pts_df.columns if col.startswith('RankforDDI')]

mapping = {1: 12, 2: 10, 3: 8, 4: 7, 5: 6, 6: 5, 7: 4, 8: 3, 9: 2, 10: 1}

for col in rank_columns:
    points_col = col.replace('RankforDDI','nPointsforDDI')
    televoting_final_pts_df[points_col] = televoting_final_pts_df[col].apply(lambda x: mapping.get(x, 0) if x <= 10 else 0)
    
televoting_final_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Moldova,nVotesforDDI02_Sweden,nVotesforDDI03_Croatia,nVotesforDDI04_Greece,nVotesforDDI05_Portugal,nVotesforDDI06_Georgia,nVotesforDDI07_Finland,nVotesforDDI08_Montenegro,nVotesforDDI09_Estonia,nVotesforDDI10_Israel,nVotesforDDI11_Belgium,nVotesforDDI12_Lithuania,nVotesforDDI13_San Marino,nVotesforDDI14_Poland,nVotesforDDI15_Serbia,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Moldova,Rankfor02_Sweden,Rankfor03_Croatia,Rankfor04_Greece,Rankfor05_Portugal,Rankfor06_Georgia,Rankfor07_Finland,Rankfor08_Montenegro,Rankfor09_Estonia,Rankfor10_Israel,Rankfor11_Belgium,Rankfor12_Lithuania,Rankfor13_San Marino,Rankfor14_Poland,Rankfor15_Serbia,RankforDDI01_Moldova,RankforDDI02_Sweden,RankforDDI03_Croatia,RankforDDI04_Greece,RankforDDI05_Portugal,RankforDDI06_Georgia,RankforDDI07_Finland,RankforDDI08_Montenegro,RankforDDI09_Estonia,RankforDDI10_Israel,RankforDDI11_Belgium,RankforDDI12_Lithuania,RankforDDI13_San Marino,RankforDDI14_Poland,RankforDDI15_Serbia,nPointsforDDI01_Moldova,nPointsforDDI02_Sweden,nPointsforDDI03_Croatia,nPointsforDDI04_Greece,nPointsforDDI05_Portugal,nPointsforDDI06_Georgia,nPointsforDDI07_Finland,nPointsforDDI08_Montenegro,nPointsforDDI09_Estonia,nPointsforDDI10_Israel,nPointsforDDI11_Belgium,nPointsforDDI12_Lithuania,nPointsforDDI13_San Marino,nPointsforDDI14_Poland,nPointsforDDI15_Serbia
0,Germany,DE,Pot00,AGGREGATED,15590,24093,13930,1161,8739,15643,11626,26926,9374,16434,1466,4504,10164,26958,21790,208398,True,Germany,12,9,8,7,5,15,10,2,1,6,3,11,14,13,4,7,3,8,15,12,6,9,2,11,5,14,13,10,1,4,4,8,3,0,0,5,2,10,0,6,0,0,1,12,7
1,Italy,IT,Pot00,AGGREGATED,38503,63217,61933,10658,12842,33438,47156,19519,46145,38232,41211,41928,10144,48391,58456,571773,True,Italy,13,6,2,5,11,14,12,7,3,15,8,9,4,1,10,9,1,2,14,13,11,5,12,6,10,8,7,15,4,3,2,12,10,0,0,0,6,0,5,1,3,4,0,7,8
2,Rest Of World,RoW,Pot00,AGGREGATED,78802,68166,9540,62557,46258,55967,39384,83818,53653,40695,29742,3214,35578,34619,8676,650669,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,3,13,4,7,5,9,1,6,8,12,15,10,11,14,10,8,0,7,4,6,2,12,5,3,0,0,1,0,0
3,Serbia,RS,Pot01,AGGREGATED,779,4003,7074,5773,9714,6880,345,2173,1399,6288,5991,4540,7567,8212,0,70738,True,Serbia,1,6,2,5,3,7,12,4,8,9,10,14,11,13,0,13,10,4,8,1,5,14,11,12,6,7,9,3,2,15,0,1,7,3,12,6,0,0,0,5,4,2,8,10,0
4,Sweden,SE,Pot01,AGGREGATED,33294,0,35829,37477,4845,20837,17985,31425,14262,15403,22973,34920,19039,9080,19913,317282,True,Sweden,8,0,3,14,11,4,5,10,1,13,2,6,9,12,7,4,15,2,1,14,7,10,5,12,11,6,3,9,13,8,7,0,10,12,0,4,1,6,0,0,5,8,2,0,3
5,Finland,FI,Pot01,AGGREGATED,12972,10320,14949,31662,41385,65238,0,43593,52671,30657,11313,26187,26403,63636,58239,489225,True,Finland,10,6,13,3,5,9,0,14,8,4,1,7,2,12,11,12,14,11,7,6,1,15,5,4,8,13,10,9,2,3,0,0,0,4,5,12,0,6,7,3,0,1,2,10,8
6,Croatia,HR,Pot01,AGGREGATED,5657,15377,0,12131,13970,11813,46008,65119,54917,52051,45205,55037,31711,50517,34019,493532,True,Croatia,3,6,0,5,4,12,14,11,2,10,8,7,13,9,1,14,10,15,12,11,13,6,1,3,4,7,2,9,5,8,0,1,0,0,0,0,5,12,8,7,4,10,2,6,3
7,Montenegro,ME,Pot01,AGGREGATED,1327,32850,49026,81409,56818,42453,33378,0,46488,58759,58707,62150,4721,40122,37559,605767,True,Montenegro,5,10,11,3,6,12,14,0,9,4,1,13,8,2,7,14,12,6,1,5,8,11,15,7,3,4,2,13,9,10,0,0,5,12,6,3,0,0,4,8,7,10,0,2,1
8,Poland,PL,Pot02,AGGREGATED,23823,30575,20883,21299,26384,15049,25028,30940,34645,29708,4116,18688,25285,0,2417,308840,True,Poland,12,8,3,4,5,10,6,11,9,7,2,1,13,0,14,8,3,10,9,5,12,7,2,1,4,13,11,6,15,14,3,8,1,2,6,0,4,10,12,7,0,0,5,0,0
9,Moldova,MD,Pot02,AGGREGATED,0,1134,6453,24289,17788,15540,38113,1164,39478,47548,27488,18049,20009,47548,6407,311008,True,Moldova,0,10,8,7,5,14,6,4,12,11,3,1,2,13,9,15,14,11,6,9,10,4,13,3,1,5,8,7,2,12,0,0,0,5,2,1,7,0,8,12,6,3,4,10,0


### Stage 5f: Compute televoting tie-breaking rank


In [30]:
############# TELEVOTING 5 : Tie break #############

# Step 1: Filter the columns
filtered_columns = [col for col in televoting_final_pts_df.columns if col.startswith("nPointsforDDI")]
filtered_df = televoting_final_pts_df[filtered_columns]

transposed_df = filtered_df.T
transposed_df.columns = televoting_final_pts_df["FullCountryName"].values
tv_tiebreaking_df = transposed_df
tv_tiebreaking_df = tv_tiebreaking_df.rename(columns={"index": "Recipient_Country"})

country_columns = tv_tiebreaking_df.columns

tv_tiebreaking_df["Pts_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row != 0).sum(), axis=1)
tv_tiebreaking_df["12_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 12).sum(), axis=1)
tv_tiebreaking_df["10_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 10).sum(), axis=1)
tv_tiebreaking_df["8_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 8).sum(), axis=1)
tv_tiebreaking_df["7_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 7).sum(), axis=1)
tv_tiebreaking_df["6_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 6).sum(), axis=1)
tv_tiebreaking_df["5_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 5).sum(), axis=1)
tv_tiebreaking_df["4_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 4).sum(), axis=1)
tv_tiebreaking_df["3_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 3).sum(), axis=1)
tv_tiebreaking_df["2_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 2).sum(), axis=1)
tv_tiebreaking_df["1_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 1).sum(), axis=1)

tv_tiebreaking_df = tv_tiebreaking_df.reset_index()
tv_tiebreaking_df = tv_tiebreaking_df.rename(columns={"index": "Country"})

# Final tie-break per EBU PDF §1.2.1: audience count → 12-pt count → 10-pt count → … → 1-pt count
# → show running order (earlier wins). The DDI prefix in 'Country' encodes the show order.
tv_tiebreaking_df["_ShowOrder"] = (
    tv_tiebreaking_df["Country"].str.extract(r"DDI(\d+)")[0].astype(int)
)
tv_tiebreaking_df = tv_tiebreaking_df.sort_values(
    by=[
        "Pts_from_X_National_Audiences",
        "12_from_X_National_Audiences",
        "10_from_X_National_Audiences",
        "8_from_X_National_Audiences",
        "7_from_X_National_Audiences",
        "6_from_X_National_Audiences",
        "5_from_X_National_Audiences",
        "4_from_X_National_Audiences",
        "3_from_X_National_Audiences",
        "2_from_X_National_Audiences",
        "1_from_X_National_Audiences",
        "_ShowOrder",
    ],
    ascending=[False, False, False, False, False, False, False, False, False, False, False, True],
)
tv_tiebreaking_df = tv_tiebreaking_df.drop(columns=["_ShowOrder"])

columns = [
    "Pts_from_X_National_Audiences",
    "12_from_X_National_Audiences",
    "10_from_X_National_Audiences",
    "8_from_X_National_Audiences",
    "7_from_X_National_Audiences",
    "6_from_X_National_Audiences",
    "6_from_X_National_Audiences",
    "5_from_X_National_Audiences",
    "4_from_X_National_Audiences",
    "3_from_X_National_Audiences",
    "2_from_X_National_Audiences",
    "1_from_X_National_Audiences",
]

# Move the specified columns right after the first column
cols = tv_tiebreaking_df.columns.tolist()
for col in reversed(columns):
    if col in cols:
        cols.insert(1, cols.pop(cols.index(col)))
tv_tiebreaking_df = tv_tiebreaking_df[cols]

# Add a new column 'TIEBREAKING_RANK'
tv_tiebreaking_df.insert(1, "TIEBREAKING_RANK", range(1, len(tv_tiebreaking_df) + 1))
tv_tiebreaking_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "05.televoting_received_pts_tiebreakers.xlsx"), index=False)

tv_tiebreaking_df

,Country,TIEBREAKING_RANK,Pts_from_X_National_Audiences,12_from_X_National_Audiences,10_from_X_National_Audiences,8_from_X_National_Audiences,7_from_X_National_Audiences,6_from_X_National_Audiences,5_from_X_National_Audiences,4_from_X_National_Audiences,3_from_X_National_Audiences,2_from_X_National_Audiences,1_from_X_National_Audiences,Germany,Italy,Rest Of World,Serbia,Sweden,Finland,Croatia,Montenegro,Poland,Moldova,Georgia,Israel,Belgium,Portugal,Lithuania,San Marino,Greece,Estonia
9,nPointsforDDI10_Israel,1,16,1,2,1,3,3,1,1,3,0,1,6,1,3,5,0,3,7,8,7,12,6,0,10,4,3,10,7,6
13,nPointsforDDI14_Poland,2,14,2,3,1,2,1,2,1,0,1,1,12,7,0,10,0,10,6,2,0,10,7,8,12,5,0,1,4,5
11,nPointsforDDI12_Lithuania,3,14,2,2,1,1,0,0,3,1,1,3,0,4,0,2,8,1,10,10,0,3,4,12,7,1,0,4,1,12
5,nPointsforDDI06_Georgia,4,14,1,1,2,0,2,2,1,2,2,1,5,0,6,6,4,12,0,3,0,1,0,10,8,2,8,5,2,3
7,nPointsforDDI08_Montenegro,5,13,3,2,0,0,2,0,1,1,2,2,10,0,12,0,6,6,12,0,10,0,2,2,1,3,4,12,0,1
2,nPointsforDDI03_Croatia,6,12,1,3,0,3,1,2,0,1,0,1,3,10,0,7,10,0,0,5,1,0,5,7,6,0,12,7,0,10
8,nPointsforDDI09_Estonia,7,12,1,0,3,1,1,2,2,1,1,0,0,5,5,0,0,7,8,4,12,8,8,4,2,0,6,0,3,0
12,nPointsforDDI13_San Marino,8,12,0,1,1,0,1,2,2,0,3,2,1,0,1,8,2,2,2,0,5,4,10,0,4,6,0,0,5,0
3,nPointsforDDI04_Greece,9,11,2,0,0,2,1,2,1,1,2,0,0,0,7,3,12,4,0,12,2,5,0,6,5,0,0,2,0,7
4,nPointsforDDI05_Portugal,10,11,1,1,0,1,2,2,1,1,2,0,0,0,4,12,0,5,0,6,6,2,0,5,3,0,7,0,10,2


### Stage 5g: Add the `TELEVOTING_PTS_TOTAL` row


In [31]:
############# TELEVOTING 5 : Tie break #############
# Add a TELEVOTING_PTS_TOTAL row summing nPointsforDDI* across every voting country.

votes_columns = [col for col in televoting_final_pts_df.columns if col.startswith("nPointsforDDI")]
new_row_sums_votes = pd.DataFrame(columns=televoting_final_pts_df.columns, index=[len(televoting_final_pts_df)])

for col in votes_columns:
    new_row_sums_votes[col] = televoting_final_pts_df[col].sum()

new_row_sums_votes["FullCountryName"] = "TELEVOTING_PTS_TOTAL"
televoting_final_pts_df = pd.concat([televoting_final_pts_df, new_row_sums_votes], ignore_index=True)

televoting_final_pts_df.to_csv(os.path.join("Output", "PP-Audience-Rank.csv"))

### Stage 5h: Build the final televoting results dataframe


In [32]:
############# TELEVOTING 5 : Final DF #############

final_televoting_results_df = televoting_final_pts_df.set_index('FullCountryName')
transposed_final_televoting_results_df = final_televoting_results_df.T
transposed_final_televoting_results_df = transposed_final_televoting_results_df.reset_index()


transposed_final_televoting_results_df = transposed_final_televoting_results_df.rename(columns={'index': 'Country'})
transposed_final_televoting_results_df = transposed_final_televoting_results_df[transposed_final_televoting_results_df['Country'].isin(votes_columns)]
transposed_final_televoting_results_df = transposed_final_televoting_results_df.sort_values(by='TELEVOTING_PTS_TOTAL', ascending=False)

# Move the 'TELEVOTING_PTS_TOTAL' column to the second position
cols = transposed_final_televoting_results_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('TELEVOTING_PTS_TOTAL')))
transposed_final_televoting_results_df = transposed_final_televoting_results_df[cols]

# Replace the substring 'nVotesforDDI' with 'nPointsforDDI' in the 'Country' column
transposed_final_televoting_results_df['Country'] = transposed_final_televoting_results_df['Country'].str.replace('nVotesforDDI', 'nPointsforDDI')

transposed_final_televoting_results_df

FullCountryName,Country,TELEVOTING_PTS_TOTAL,Germany,Italy,Rest Of World,Serbia,Sweden,Finland,Croatia,Montenegro,Poland,Moldova,Georgia,Israel,Belgium,Portugal,Lithuania,San Marino,Greece,Estonia
64,nPointsforDDI14_Poland,99,12,7,0,10,0,10,6,2,0,10,7,8,12,5,0,1,4,5
60,nPointsforDDI10_Israel,98,6,1,3,5,0,3,7,8,7,12,6,0,10,4,3,10,7,6
53,nPointsforDDI03_Croatia,83,3,10,0,7,10,0,0,5,1,0,5,7,6,0,12,7,0,10
58,nPointsforDDI08_Montenegro,81,10,0,12,0,6,6,12,0,10,0,2,2,1,3,4,12,0,1
62,nPointsforDDI12_Lithuania,79,0,4,0,2,8,1,10,10,0,3,4,12,7,1,0,4,1,12
52,nPointsforDDI02_Sweden,78,8,12,8,1,0,0,1,0,8,0,12,0,0,12,0,8,0,8
56,nPointsforDDI06_Georgia,75,5,0,6,6,4,12,0,3,0,1,0,10,8,2,8,5,2,3
59,nPointsforDDI09_Estonia,72,0,5,5,0,0,7,8,4,12,8,8,4,2,0,6,0,3,0
54,nPointsforDDI04_Greece,65,0,0,7,3,12,4,0,12,2,5,0,6,5,0,0,2,0,7
55,nPointsforDDI05_Portugal,62,0,0,4,12,0,5,0,6,6,2,0,5,3,0,7,0,10,2


### Stage 5i: Attach televoting tie-breaking rank and final 1..N rank


In [33]:
############# TELEVOTING 6 : Final DF #############

transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_df
tv_tiebreaking_ranks_only_df = tv_tiebreaking_df[['Country', 'TIEBREAKING_RANK']]
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df.merge(tv_tiebreaking_ranks_only_df, on='Country', how='left')

# Move the 'TELEVOTING_PTS_TOTAL' column to the second position
cols = transposed_final_televoting_results_w_tiebreakers_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('TIEBREAKING_RANK')))
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df[cols]

# Sort for final order
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df.sort_values(by=['TELEVOTING_PTS_TOTAL', 'TIEBREAKING_RANK'], ascending=[False, True])
transposed_final_televoting_results_w_tiebreakers_df.insert(0, 'Final_Rank', range(1, len(transposed_final_televoting_results_w_tiebreakers_df) + 1))

# Save
transposed_final_televoting_results_w_tiebreakers_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, '06.televoting_final_televoting_results_w_tiebreaker.xlsx'), index=False)

transposed_final_televoting_results_w_tiebreakers_df

,Final_Rank,Country,TELEVOTING_PTS_TOTAL,TIEBREAKING_RANK,Germany,Italy,Rest Of World,Serbia,Sweden,Finland,Croatia,Montenegro,Poland,Moldova,Georgia,Israel,Belgium,Portugal,Lithuania,San Marino,Greece,Estonia
0,1,nPointsforDDI14_Poland,99,2,12,7,0,10,0,10,6,2,0,10,7,8,12,5,0,1,4,5
1,2,nPointsforDDI10_Israel,98,1,6,1,3,5,0,3,7,8,7,12,6,0,10,4,3,10,7,6
2,3,nPointsforDDI03_Croatia,83,6,3,10,0,7,10,0,0,5,1,0,5,7,6,0,12,7,0,10
3,4,nPointsforDDI08_Montenegro,81,5,10,0,12,0,6,6,12,0,10,0,2,2,1,3,4,12,0,1
4,5,nPointsforDDI12_Lithuania,79,3,0,4,0,2,8,1,10,10,0,3,4,12,7,1,0,4,1,12
5,6,nPointsforDDI02_Sweden,78,13,8,12,8,1,0,0,1,0,8,0,12,0,0,12,0,8,0,8
6,7,nPointsforDDI06_Georgia,75,4,5,0,6,6,4,12,0,3,0,1,0,10,8,2,8,5,2,3
7,8,nPointsforDDI09_Estonia,72,7,0,5,5,0,0,7,8,4,12,8,8,4,2,0,6,0,3,0
8,9,nPointsforDDI04_Greece,65,9,0,0,7,3,12,4,0,12,2,5,0,6,5,0,0,2,0,7
9,10,nPointsforDDI05_Portugal,62,10,0,0,4,12,0,5,0,6,6,2,0,5,3,0,7,0,10,2


### Quick inspection of the final televoting results


In [34]:
transposed_final_televoting_results_w_tiebreakers_df

,Final_Rank,Country,TELEVOTING_PTS_TOTAL,TIEBREAKING_RANK,Germany,Italy,Rest Of World,Serbia,Sweden,Finland,Croatia,Montenegro,Poland,Moldova,Georgia,Israel,Belgium,Portugal,Lithuania,San Marino,Greece,Estonia
0,1,nPointsforDDI14_Poland,99,2,12,7,0,10,0,10,6,2,0,10,7,8,12,5,0,1,4,5
1,2,nPointsforDDI10_Israel,98,1,6,1,3,5,0,3,7,8,7,12,6,0,10,4,3,10,7,6
2,3,nPointsforDDI03_Croatia,83,6,3,10,0,7,10,0,0,5,1,0,5,7,6,0,12,7,0,10
3,4,nPointsforDDI08_Montenegro,81,5,10,0,12,0,6,6,12,0,10,0,2,2,1,3,4,12,0,1
4,5,nPointsforDDI12_Lithuania,79,3,0,4,0,2,8,1,10,10,0,3,4,12,7,1,0,4,1,12
5,6,nPointsforDDI02_Sweden,78,13,8,12,8,1,0,0,1,0,8,0,12,0,0,12,0,8,0,8
6,7,nPointsforDDI06_Georgia,75,4,5,0,6,6,4,12,0,3,0,1,0,10,8,2,8,5,2,3
7,8,nPointsforDDI09_Estonia,72,7,0,5,5,0,0,7,8,4,12,8,8,4,2,0,6,0,3,0
8,9,nPointsforDDI04_Greece,65,9,0,0,7,3,12,4,0,12,2,5,0,6,5,0,0,2,0,7
9,10,nPointsforDDI05_Portugal,62,10,0,0,4,12,0,5,0,6,6,2,0,5,3,0,7,0,10,2


## Final jury results (recap)


In [35]:
##########################
### FINAL JURY RESULTS ###
##########################

transposed_final_jury_results_w_tiebreakers_df

,Final_Rank,Recipient_Country,Total_without_RoW,TIEBREAKING_RANK,Total_with_RoW,Moldova,Sweden,Croatia,Greece,Portugal,Georgia,Finland,Montenegro,Estonia,Israel,Belgium,Lithuania,San Marino,Poland,Serbia,Germany,Italy,Rest of World,Rank_with_RoW,Rank_without_RoW
0,1,nPointsforDDI11_Belgium,87,1.0,93,8,10,3,7,3,4,12,12,0,1,0,0,5,10,1,8,3,6,NaN,NaN
1,2,nPointsforDDI15_Serbia,82,4.0,86,2,4,12,6,7,0,0,4,7,4,10,12,6,0,0,7,1,4,NaN,NaN
2,3,nPointsforDDI04_Greece,79,2.0,84,4,0,6,0,10,8,8,8,1,2,0,8,1,7,6,4,6,5,NaN,NaN
5,4,nPointsforDDI09_Estonia,78,3.0,90,0,12,10,12,4,0,3,2,0,0,4,4,2,2,3,12,8,12,NaN,NaN
3,5,nPointsforDDI01_Moldova,78,5.0,78,0,3,8,8,2,5,1,6,3,5,7,10,8,0,12,0,0,0,NaN,NaN
4,6,nPointsforDDI05_Portugal,78,6.0,80,6,0,7,3,0,6,6,5,12,8,0,2,3,6,8,6,0,2,NaN,NaN
6,7,nPointsforDDI12_Lithuania,73,11.0,73,12,5,4,2,0,12,4,0,0,12,8,0,0,12,0,0,2,0,NaN,NaN
7,8,nPointsforDDI08_Montenegro,69,10.0,79,7,1,0,0,12,10,0,0,4,7,1,6,0,0,7,10,4,10,NaN,NaN
8,9,nPointsforDDI07_Finland,67,8.0,67,5,6,0,5,6,7,0,0,8,3,6,3,12,5,0,1,0,0,NaN,NaN
9,10,nPointsforDDI03_Croatia,65,9.0,72,3,8,0,1,1,2,0,0,10,0,5,0,4,8,10,3,10,7,NaN,NaN


## Manual instruction placeholder


In [36]:
### Instruction XYZ

## Grand Final calculations

When the selected dataframe is the Grand Final, combine jury (without RoW) and televoting totals into a single sorted scoreboard, and export both the simple and enriched Excel deliverables.


In [37]:
################################
### GRAND FINAL CALCULATIONS ###
################################


# Check if we currently deal with the grand finale
if selected_df_name in ["sf1_televoting_df","sf2_televoting_df",'gf_televoting_df']:
    print(f'Final Score event detected, running analysis... (currently selected: {selected_df_name})')
    
    gf_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df[['Country','TELEVOTING_PTS_TOTAL','TIEBREAKING_RANK']]
    gf_televoting_results_w_tiebreakers_df = gf_televoting_results_w_tiebreakers_df.rename(columns={'Country': 'Recipient_Country', 'TIEBREAKING_RANK': 'TELEVOTING_TIEBREAKING_RANK'})

    gf_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df[['Recipient_Country','Total_without_RoW','TIEBREAKING_RANK']]
    gf_jury_results_w_tiebreakers_df = gf_jury_results_w_tiebreakers_df.rename(columns={'Total_without_RoW': 'JURY_PTS_TOTAL_WO_ROW','TIEBREAKING_RANK': 'JURY_TIEBREAKING_RANK'})
    gf_jury_results_w_tiebreakers_df = gf_jury_results_w_tiebreakers_df[gf_jury_results_w_tiebreakers_df['Recipient_Country'].str.startswith('nPointsforDDI')]

    merged_df = gf_televoting_results_w_tiebreakers_df.merge(gf_jury_results_w_tiebreakers_df, on='Recipient_Country')
    merged_df['ALL_PTS_TOTAL'] = merged_df['TELEVOTING_PTS_TOTAL'] + merged_df['JURY_PTS_TOTAL_WO_ROW']
    merged_df.insert(1, 'ALL_PTS_TOTAL', merged_df.pop('ALL_PTS_TOTAL'))
    merged_df = merged_df.sort_values(['ALL_PTS_TOTAL','TELEVOTING_PTS_TOTAL'], ascending=False)
    gf_final_merged_results_df = merged_df
else:
    print(f'Currently selected DF is not for Grand Finale (currently selected: {selected_df_name})')


gf_final_merged_results_df[['Recipient_Country','JURY_PTS_TOTAL_WO_ROW','TELEVOTING_PTS_TOTAL','ALL_PTS_TOTAL']].to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07a.SF1_FINAL_RESULTS_SIMPLE.xlsx'
), index=False)

gf_final_merged_results_df.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07b.SF1_FINAL_RESULTS_ENRICHED.xlsx'
), index=False)


#gf_final_merged_results_df
gf_final_merged_results_df[['Recipient_Country','JURY_PTS_TOTAL_WO_ROW','TELEVOTING_PTS_TOTAL','ALL_PTS_TOTAL']]

Final Score event detected, running analysis... (currently selected: sf1_televoting_df)


,Recipient_Country,JURY_PTS_TOTAL_WO_ROW,TELEVOTING_PTS_TOTAL,ALL_PTS_TOTAL
0,nPointsforDDI14_Poland,54,99,153
4,nPointsforDDI12_Lithuania,73,79,152
3,nPointsforDDI08_Montenegro,69,81,150
7,nPointsforDDI09_Estonia,78,72,150
2,nPointsforDDI03_Croatia,65,83,148
10,nPointsforDDI11_Belgium,87,60,147
8,nPointsforDDI04_Greece,79,65,144
9,nPointsforDDI05_Portugal,78,62,140
1,nPointsforDDI10_Israel,37,98,135
11,nPointsforDDI01_Moldova,78,56,134


## Combined final results table

Build a single combined dataframe holding, for each recipient country:

- the jury total (without RoW),
- the televoting total,
- a `FINAL_TOTAL_SCORE` summing both,
- per-donor breakdowns (`*_JURY`, `*_TV`, `*_TOTAL`).


In [38]:
###################################
### COMBINED FINAL RESULTS TABLE ##
###################################

# Build a combined dataframe from the jury and televoting tiebreaker tables,
# adding a final total score that sums points assigned by the jury and the televoting,
# and including the per-country breakdown (sum of jury + televoting points from each donor country).

# --- Identify donor-country columns in each source dataframe ---
jury_meta_cols = [
    'Final_Rank', 'Recipient_Country', 'Total_without_RoW', 'TIEBREAKING_RANK',
    'Total_with_RoW', 'Rank_with_RoW', 'Rank_without_RoW'
]
tv_meta_cols = [
    'Final_Rank', 'Country', 'TELEVOTING_PTS_TOTAL', 'TIEBREAKING_RANK'
]

jury_country_cols = [c for c in transposed_final_jury_results_w_tiebreakers_df.columns if c not in jury_meta_cols]
tv_country_cols   = [c for c in transposed_final_televoting_results_w_tiebreakers_df.columns if c not in tv_meta_cols]

# --- Prepare jury side ---
jury_side_df = transposed_final_jury_results_w_tiebreakers_df.copy()
jury_side_df = jury_side_df[jury_side_df['Recipient_Country'].str.startswith('nPointsforDDI')]
jury_side_df = jury_side_df[['Recipient_Country', 'Total_without_RoW'] + jury_country_cols].rename(
    columns={'Total_without_RoW': 'JURY_PTS_TOTAL'}
)
# Normalize 'Rest Of World' -> 'Rest Of World' so jury and televoting columns align
jury_side_df = jury_side_df.rename(columns={'Rest Of World': 'Rest Of World'})
#jury_side_df['Rest Of World'] = 0

# --- Prepare televoting side ---
tv_side_df = transposed_final_televoting_results_w_tiebreakers_df.copy()
tv_side_df = tv_side_df[['Country', 'TELEVOTING_PTS_TOTAL'] + tv_country_cols].rename(
    columns={'Country': 'Recipient_Country'}
)

# --- Merge on Recipient_Country with suffixes to keep both donor breakdowns ---
combined_final_results_df = jury_side_df.merge(
    tv_side_df, on='Recipient_Country', how='inner', suffixes=('_JURY', '_TV')
)

combined_final_results_df['FINAL_TOTAL_SCORE'] = (
    combined_final_results_df['JURY_PTS_TOTAL'] + combined_final_results_df['TELEVOTING_PTS_TOTAL']
)

# --- Build per-country combined columns: sum of jury + televoting points from each donor country ---
donor_countries = sorted(set([c.replace('Rest Of World', 'Rest Of World') for c in jury_country_cols]) | set(tv_country_cols))
for donor in donor_countries:
    j_col = donor if donor in combined_final_results_df.columns else f'{donor}_JURY'
    t_col = donor if donor in combined_final_results_df.columns else f'{donor}_TV'
    j_vals = pd.to_numeric(combined_final_results_df[j_col], errors='coerce').fillna(0) if j_col in combined_final_results_df.columns else 0
    t_vals = pd.to_numeric(combined_final_results_df[t_col], errors='coerce').fillna(0) if t_col in combined_final_results_df.columns else 0
    combined_final_results_df[f'{donor}_TOTAL'] = j_vals + t_vals

# --- Sort and rank ---
combined_final_results_df = combined_final_results_df.sort_values(
    by=['FINAL_TOTAL_SCORE', 'TELEVOTING_PTS_TOTAL'], ascending=[False, False]
).reset_index(drop=True)
combined_final_results_df.insert(0, 'Final_Rank', range(1, len(combined_final_results_df) + 1))

# --- Reorder: meta columns first, then per-country totals, then jury/televoting breakdowns ---
total_country_cols = [f'{c}_TOTAL' for c in donor_countries]
jury_only_cols = [c for c in combined_final_results_df.columns if c.endswith('_JURY')]
tv_only_cols   = [c for c in combined_final_results_df.columns if c.endswith('_TV')]
shared_cols    = [c for c in donor_countries if c in combined_final_results_df.columns]

ordered_cols = (
    ['Final_Rank', 'Recipient_Country', 'FINAL_TOTAL_SCORE', 'JURY_PTS_TOTAL', 'TELEVOTING_PTS_TOTAL']
    + total_country_cols
    + jury_only_cols + tv_only_cols + shared_cols
)
ordered_cols = [c for c in ordered_cols if c in combined_final_results_df.columns]
combined_final_results_df = combined_final_results_df[ordered_cols]

combined_final_results_df

,Final_Rank,Recipient_Country,FINAL_TOTAL_SCORE,JURY_PTS_TOTAL,TELEVOTING_PTS_TOTAL,Belgium_TOTAL,Croatia_TOTAL,Estonia_TOTAL,Finland_TOTAL,Georgia_TOTAL,Germany_TOTAL,Greece_TOTAL,Israel_TOTAL,Italy_TOTAL,Lithuania_TOTAL,Moldova_TOTAL,Montenegro_TOTAL,Poland_TOTAL,Portugal_TOTAL,Rest Of World_TOTAL,Rest of World_TOTAL,San Marino_TOTAL,Serbia_TOTAL,Sweden_TOTAL,Moldova_JURY,Sweden_JURY,Croatia_JURY,Greece_JURY,Portugal_JURY,Georgia_JURY,Finland_JURY,Montenegro_JURY,Estonia_JURY,Israel_JURY,Belgium_JURY,Lithuania_JURY,San Marino_JURY,Poland_JURY,Serbia_JURY,Germany_JURY,Italy_JURY,Germany_TV,Italy_TV,Serbia_TV,Sweden_TV,Finland_TV,Croatia_TV,Montenegro_TV,Poland_TV,Moldova_TV,Georgia_TV,Israel_TV,Belgium_TV,Portugal_TV,Lithuania_TV,San Marino_TV,Greece_TV,Estonia_TV,Rest Of World,Rest of World
0,1,nPointsforDDI14_Poland,153,54,99,14,8,10,10,7,12,4,8,19,5,10,12,0,13,0,16,11,10,0,0,0,2,0,8,0,0,10,5,0,2,5,10,0,0,0,12,12,7,10,0,10,6,2,0,10,7,8,12,5,0,1,4,5,0,8
1,2,nPointsforDDI12_Lithuania,152,73,79,15,14,12,5,16,0,3,24,6,0,15,10,12,1,0,0,4,2,13,12,5,4,2,0,12,4,0,0,12,8,0,0,12,0,0,2,0,4,2,8,1,10,10,0,3,4,12,7,1,0,4,1,12,0,0
2,3,nPointsforDDI08_Montenegro,150,69,81,2,12,5,6,12,20,0,9,4,10,7,0,10,15,24,20,12,7,7,7,1,0,0,12,10,0,0,4,7,1,6,0,0,7,10,4,10,0,0,6,6,12,0,10,0,2,2,1,3,4,12,0,1,12,10
3,4,nPointsforDDI09_Estonia,150,78,72,6,18,0,10,8,12,15,4,13,10,8,6,14,4,10,24,2,3,12,0,12,10,12,4,0,3,2,0,0,4,4,2,2,3,12,8,0,5,0,0,7,8,4,12,8,8,4,2,0,6,0,3,0,5,12
4,5,nPointsforDDI03_Croatia,148,65,83,11,0,20,0,7,6,1,7,20,12,3,5,9,1,0,14,11,17,18,3,8,0,1,1,2,0,0,10,0,5,0,4,8,10,3,10,3,10,7,10,0,0,5,1,0,5,7,6,0,12,7,0,10,0,7
5,6,nPointsforDDI11_Belgium,147,87,60,0,7,0,12,7,8,19,4,6,5,14,19,10,11,0,12,5,5,15,8,10,3,7,3,4,12,12,0,1,0,0,5,10,1,8,3,0,3,4,5,0,4,7,0,6,3,3,0,8,5,0,12,0,0,6
6,7,nPointsforDDI04_Greece,144,79,65,5,6,8,12,8,4,0,8,6,8,9,20,9,10,14,10,3,9,12,4,0,6,0,10,8,8,8,1,2,0,8,1,7,6,4,6,0,0,3,12,4,0,12,2,5,0,6,5,0,0,2,0,7,7,5
7,8,nPointsforDDI05_Portugal,140,78,62,3,7,14,11,6,6,13,13,0,9,8,11,12,0,8,4,3,20,0,6,0,7,3,0,6,6,5,12,8,0,2,3,6,8,6,0,0,0,12,0,5,0,6,6,2,0,5,3,0,7,0,10,2,4,2
8,9,nPointsforDDI10_Israel,135,37,98,13,8,6,10,7,11,7,0,1,10,12,15,11,4,6,0,10,7,0,0,0,1,0,0,1,7,7,0,0,3,7,0,4,2,5,0,6,1,5,0,3,7,8,7,12,6,0,10,4,3,10,7,6,3,0
9,10,nPointsforDDI01_Moldova,134,78,56,7,8,7,1,5,4,16,5,2,12,0,6,3,12,20,0,14,12,10,0,3,8,8,2,5,1,6,3,5,7,10,8,0,12,0,0,4,2,0,7,0,0,0,3,0,0,0,0,10,2,6,8,4,10,0


### Export the combined final results to Excel


In [39]:
combined_final_results_df.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07b.SF1_FINAL_RESULTS_ENRICHED-all.xlsx'
), index=False)